### Implementing speech augmentation
Based on Minh's wav2vec2 audio embedding generation

In [4]:
# !pip install speechbrain

In [65]:
from speechbrain.augment.time_domain import AddNoise, DropFreq, SpeedPerturb, RandAmp
from speechbrain.augment.freq_domain import Warping
from speechbrain.augment.augmenter import Augmenter
from speechbrain.dataio.dataio import read_audio, write_audio
import torch
import numpy as np
import os

In [66]:
paths = [
    "../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/",
    "../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/",
    "../data/Test Dataset/Diagnosis Testing dataset/diagnosis/test-dist/audio/",
]

In [67]:
def get_file_list(path, ext=".wav"):
    """
    get all filenames in a folder
    returns a dictionary {file_id:full_path}
    """
    result = dict()
    print("Reading from folder =", path)
    files = [item for item in os.listdir(path) if item.endswith(ext)]

    for fname in files:
        result[fname.replace(ext, "")] = path + fname

    print("Count =", len(result))
    return result

In [68]:
file_paths_dict = [get_file_list(path) for path in paths]

Reading from folder = ../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/ad/
Count = 87
Reading from folder = ../data/Transcribed Data-2024-Feb-13/Diagnosis Training Data/cn/
Count = 79
Reading from folder = ../data/Test Dataset/Diagnosis Testing dataset/diagnosis/test-dist/audio/
Count = 71


In [69]:
# merge dictionaries
file_paths_dict = dict()

for fpath in file_paths:
    file_paths_dict.update(fpath)

print(len(file_paths_dict))

237


In [71]:
# https://github.com/speechbrain/speechbrain/blob/develop/speechbrain/augment/augmenter.py
freq_dropper = DropFreq()
noisifier = AddNoise()
perturbator = SpeedPerturb(orig_freq=16000, speeds=[100])
rand_amp = RandAmp(amp_low=0.25, amp_high=1.75)
# warp = Warping()
augment = Augmenter(
    parallel_augment=False,
    concat_original=False,
    augmentations=[noisifier, freq_dropper, perturbator],
)

In [73]:
for k, v in file_paths_dict.items():
    signal = read_audio(v).unsqueeze(0)
    print("Original signal:", signal)
    print(
        "Augmented signal: ",
        augment(signal, lengths=torch.tensor([0.2, 0.5, 0.7, 1.0])),
    )
    break

Original signal: tensor([[-3.8452e-03,  4.0588e-03, -2.2583e-03,  ...,  0.0000e+00,
         -6.1035e-05,  0.0000e+00]])
Augmented signal:  (tensor([[-0.0417, -0.0391, -0.0052,  ...,  0.0284,  0.0771, -0.0392]]), tensor([0.2000]))
